# Feature Families + Binary Target Engineering Matrix

This notebook checks whether Quant Warehouse can combine reusable feature families with binary target-engineering events. It focuses on joinability, target sparsity, and family-level signal separation rather than Rank IC.

Targets covered here:

- FMP event pairs: congress buy/sell, insider buy/sell, analyst upgrades/downgrades, price target raises/cuts, guidance raises/cuts, earnings beats/misses.
- Oracle trade entry labels from stored prices: yearly top-k long/short entries using the batched optimal-trade solver with `YE: 1..12`. Execution prices match `optimal_trader`: buy at high, sell at low, short at low, cover at high.

The goal is to answer: for each feature family, which binary target families have enough joined rows to be worth deeper modeling?

In [ ]:
from __future__ import annotations

from pathlib import Path
from time import perf_counter
import sys

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'quant_warehouse').exists():
    REPO_ROOT = next(parent for parent in Path.cwd().parents if (parent / 'quant_warehouse').exists())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
from IPython.display import Markdown, display

from quant_warehouse.platforms.data_providers.fmp.target_engineering.event_pairs import EventPairStore
from quant_warehouse.research_tools import (
    BinaryTargetConfig,
    FamilyEvaluationConfig,
    build_event_target_panel,
    build_fundamental_feature_panel,
    build_oracle_trade_target_panel,
    cap_features_by_quality,
    combine_target_panels,
    evaluate_feature_target_matrix,
    load_fmp_event_pairs,
    screen_fmp_equity_universe,
    summarize_binary_targets,
)
from quant_warehouse.warehouse.api import Warehouse

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 180)

In [ ]:
RUN_STARTED = perf_counter()

FEATURE_CONFIG = FamilyEvaluationConfig(
    provider='fmp',
    market_cap_min=100_000_000_000,
    start_date='2018-01-01',
    horizons=(20, 60),
    min_observations=120,
    max_features_per_family=50,
)

TARGET_CONFIG = BinaryTargetConfig(
    provider='fmp',
    start_date=FEATURE_CONFIG.start_date,
    end_date=FEATURE_CONFIG.end_date,
    event_families=(
        'congress',
        'insider',
        'analyst_rating',
        'price_target',
        'guidance',
        'earnings',
    ),
    oracle_trade_k_by_frequency={'YE': tuple(range(1, 13))},
    oracle_trade_min_profit_pct=0.01,
    oracle_trade_long_entry_price_col='high',  # buy execution
    oracle_trade_long_exit_price_col='low',    # sell execution
    oracle_trade_short_entry_price_col='low',  # short execution
    oracle_trade_short_exit_price_col='high',  # cover execution
)

FEATURE_CONFIG, TARGET_CONFIG

WAREHOUSE = Warehouse()
EVENT_STORE = EventPairStore(
    config=WAREHOUSE.config,
    backend=WAREHOUSE.backend,
    catalog=WAREHOUSE.catalog,
    fundamentals=WAREHOUSE.fundamentals,
    equity_calendar=WAREHOUSE.equity_calendar,
)

## Universe and Feature Families

The notebook starts with FMP equities screened through the OpenBB/FMP screener route, then keeps symbols with the locally stored sections required for the fundamental feature families.

In [ ]:
symbols, raw_universe, eligibility, universe_source = screen_fmp_equity_universe(FEATURE_CONFIG, warehouse=WAREHOUSE)

print(f'Universe source: {universe_source}')
print(f'Eligible symbols: {len(symbols)}')
print(symbols[:50])

eligibility.head(20)

In [ ]:
raw_feature_panel, raw_feature_metadata, feature_diagnostics, feature_timings = build_fundamental_feature_panel(
    symbols,
    FEATURE_CONFIG,
    warehouse=WAREHOUSE,
)

print({
    'screened_symbols': len(symbols),
    'raw_feature_panel_rows': len(raw_feature_panel),
    'raw_features': raw_feature_metadata['feature'].nunique(),
    **feature_timings,
})

feature_diagnostics.sort_values(['status', 'rows'], ascending=[True, False]).head(20)

## Binary Event Targets

Event targets are loaded without remote refresh. Cached normalized event-pair data is used when present; missing families are built from already stored FMP historical sections.

In [ ]:
events, event_diagnostics, event_load_seconds = load_fmp_event_pairs(
    symbols,
    TARGET_CONFIG,
    event_store=EVENT_STORE,
    include_historical=True,
)

event_symbols = tuple(
    event_diagnostics
    .loc[event_diagnostics['combined_rows'].gt(0), 'symbol']
    .sort_values()
    .tolist()
)
feature_panel = raw_feature_panel.loc[raw_feature_panel['symbol'].isin(event_symbols)].copy()
feature_metadata = raw_feature_metadata.copy()
selected_features, capped_feature_metadata, feature_quality = cap_features_by_quality(
    feature_panel,
    feature_metadata,
    max_features=FEATURE_CONFIG.max_features_per_family,
)
feature_inventory = (
    capped_feature_metadata
    .groupby(['source', 'family'])
    .agg(feature_count=('feature', 'nunique'))
    .reset_index()
    .sort_values(['source', 'family'])
)

event_target_panel, event_target_metadata = build_event_target_panel(
    feature_panel[['symbol', 'date']],
    events,
    TARGET_CONFIG,
)

print({
    'screened_symbols': len(symbols),
    'event_symbols': len(event_symbols),
    'event_rows': len(events),
    'feature_panel_rows_after_event_symbol_filter': len(feature_panel),
    'capped_features': len(selected_features),
    'event_target_rows': len(event_target_panel),
    'event_target_columns': len(event_target_metadata),
    'event_load_seconds': round(event_load_seconds, 3),
})

display(feature_inventory)
event_diagnostics.sort_values('combined_rows', ascending=False).head(20)

In [ ]:
oracle_target_panel, oracle_target_metadata, oracle_seconds = build_oracle_trade_target_panel(
    event_symbols,
    TARGET_CONFIG,
    warehouse=WAREHOUSE,
)

target_panel = combine_target_panels(event_target_panel, oracle_target_panel)
target_metadata = pd.concat([event_target_metadata, oracle_target_metadata], ignore_index=True)
target_summary = summarize_binary_targets(target_panel, target_metadata)

print({
    'oracle_target_rows': len(oracle_target_panel),
    'oracle_target_columns': len(oracle_target_metadata),
    'oracle_seconds': round(oracle_seconds, 3),
    'combined_target_rows': len(target_panel),
    'combined_target_columns': target_metadata['target'].nunique(),
})

target_summary

## Feature Family x Target Matrix

This matrix asks whether a model row can be formed for each pair. `mean_abs_smd` is a fast family-level separation check: larger values mean the positive and negative target rows look more different on average. It is not a full trading evaluation.

In [ ]:
matrix, training_panel = evaluate_feature_target_matrix(
    feature_panel,
    capped_feature_metadata,
    target_panel,
    target_metadata,
    min_rows=120,
    min_positive_rows=10,
    min_feature_coverage=0.5,
)

usable = matrix.query("status == 'usable'").copy()

print({
    'matrix_rows': len(matrix),
    'usable_pairs': len(usable),
    'training_panel_rows': len(training_panel),
    'elapsed_seconds': round(perf_counter() - RUN_STARTED, 3),
})

usable.head(50)

In [ ]:
coverage_pivot = (
    matrix
    .pivot_table(
        index=['source', 'feature_family'],
        columns='target_family',
        values='target',
        aggfunc='count',
        fill_value=0,
    )
    .reset_index()
)

positive_pivot = (
    usable
    .pivot_table(
        index=['source', 'feature_family'],
        columns='target_family',
        values='positive_rows',
        aggfunc='max',
        fill_value=0,
    )
    .reset_index()
)

coverage_pivot

In [ ]:
best_by_target = (
    usable
    .sort_values(['target', 'mean_abs_smd', 'positive_rows'], ascending=[True, False, False])
    .groupby('target')
    .head(3)
    .reset_index(drop=True)
)

best_by_target

In [ ]:
sparse_targets = target_summary.query('positive_rows < 10').copy()
usable_targets = usable['target'].nunique() if not usable.empty else 0
usable_feature_families = usable['feature_family'].nunique() if not usable.empty else 0
best_pairs = usable.sort_values('mean_abs_smd', ascending=False).head(10) if not usable.empty else usable

lines = [
    '## Written Analysis',
    '',
    f'- Universe: {len(symbols)} screened FMP symbols with market cap >= ${FEATURE_CONFIG.market_cap_min:,.0f}; {len(event_symbols)} symbols had at least one loaded event pair and were used in the target matrix.',
    f'- Feature side: {capped_feature_metadata["family"].nunique()} capped feature families and {len(selected_features)} selected feature columns.',
    f'- Target side: {target_metadata["target"].nunique()} binary target columns across same-day event and oracle-trade families.',
    f'- Joinability: {len(usable)} feature-family/target pairs are usable under the current thresholds, covering {usable_targets} target columns and {usable_feature_families} feature families.',
]

if not sparse_targets.empty:
    lines.append(f'- Sparse targets: {len(sparse_targets)} target columns have fewer than 10 positive rows and should not be trusted for modeling yet.')

if not best_pairs.empty:
    lines.append('')
    lines.append('Top family/target combinations by fast separation score:')
    for row in best_pairs[['source', 'feature_family', 'target', 'positive_rows', 'mean_abs_smd']].to_dict('records'):
        lines.append(
            f'- {row["source"]}.{row["feature_family"]} -> {row["target"]}: '
            f'{int(row["positive_rows"])} positives, mean_abs_smd={row["mean_abs_smd"]:.4f}'
        )

lines.extend([
    '',
    'Interpretation:',
    '',
    '- A usable pair only means the warehouse can form a training matrix with enough positive binary events. It does not prove tradeability.',
    '- Oracle-trade labels are sparse yearly top-k entry labels from the batched optimal-trade solver; they match the fast `optimal_trader` style better than dense daily horizon labels.',
    '- Congress, insider, guidance, and earnings labels are same-day sparse events and should be interpreted at the family level first before actor-specific or event-specific modeling.',
    '- The next practical step is to choose a few usable target families, then add leakage-safe event-date rules and transaction-cost-aware evaluation in Quant Orchestrator.',
])

display(Markdown('\n'.join(lines)))